# Appending the 'appended_data.dta' dataset: Google Scholar values update
**Note:** For reproducibility:



1.   First make a copy of the notebook and work on the copy
2.   Check these things: paths name, file names, APIs IDs, and customize them according to your environment
3.   After doing a major data operation, you can push your final version here: https://github.com/GaelKBertrand/scholar_info_append
4.   Notify colleagues for any data changes so that everyone can be on the same page








### Mounting Google Drive as a working director

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Scrapping

In [ ]:
#Install dependencies
from requests_html import HTMLSession
import pandas as pd
import time

# Initialize session
session = HTMLSession()

# Load your master dataset from MyDrive or from where you placed it in your google drive
df = pd.read_csv('/content/drive/My Drive/clean/merge/master_dataset.csv')  # Update the file name also accordingly

# Loop through each row in the dataset, the row of scholar profile links is 'scholar_profile_url'
for idx, row in df.iterrows():
    url = row['scholar_profile_url']

    # Skip empty or 'not found' URLs
    if pd.isna(url) or url == 'not found':
        print(f"Skipping row {idx} as URL is empty or 'not found'.")
        continue

    try:
        # Send a request to the URL
        r = session.get(url)

        # Extract the required information from the page

        # Name
        name = r.html.find('#gsc_prf_in', first=True).text

        # Citation-related data
        total_citation = r.html.find('.gsc_rsb_std', first=True).text
        citations_dynmc = r.html.find('#gsc_rsb_st > tbody > tr:nth-child(1) > td:nth-child(3)', first=True).text
        h_index = r.html.find('#gsc_rsb_st > tbody > tr:nth-child(2) > td:nth-child(2)', first=True).text
        h_index_5y_dynmc = r.html.find('#gsc_rsb_st > tbody > tr:nth-child(2) > td:nth-child(3)', first=True).text
        i10_index = r.html.find('#gsc_rsb_st > tbody > tr:nth-child(3) > td:nth-child(2)', first=True).text
        i10_index_5y_dynmc = r.html.find('#gsc_rsb_st > tbody > tr:nth-child(3) > td:nth-child(3)', first=True).text

        # Scrape position
        position_element = r.html.find('div#gsc_prf_i > div:nth-child(2)', first=True)
        position = position_element.text if position_element else 'not found'

        # Scrape field
        fields_elements = r.html.find('div#gsc_prf_int > a')
        field = ', '.join([field.text for field in fields_elements]) if fields_elements else 'not found'

        # Scrape co-authors
        co_authors_elements = r.html.find('#gsc_rsb_co')
        co_authors = 'not found'
        if co_authors_elements:
            co_authors_text = co_authors_elements[0].text
            co_authors_list = [line.split('\n')[0] for line in co_authors_text.split('\n') if line.strip() and not line.startswith('#')]
            if co_authors_list and co_authors_list[0].lower().startswith('co-authors'):
                co_authors_list = co_authors_list[1:]
            if co_authors_list:
                co_authors = ', '.join(co_authors_list)

        # Update the master dataset (df) with the new data
        df.at[idx, 'total_citation'] = total_citation
        df.at[idx, 'citations_dynmc'] = citations_dynmc
        df.at[idx, 'h-index'] = h_index
        df.at[idx, 'h-index_5y_dynmc'] = h_index_5y_dynmc
        df.at[idx, 'i10-index'] = i10_index
        df.at[idx, 'i10-index_5y_dynmc'] = i10_index_5y_dynmc
        df.at[idx, 'position'] = position
        df.at[idx, 'field'] = field
        df.at[idx, 'co-authors'] = co_authors

        # Print the name processed
        print(f"Processed {name}")

    except Exception as e:
        print(f"Error processing row {idx} with URL {url}: {e}")
        # Skip this row on error, or handle it as needed

# Save the updated master dataset back to MyDrive
df.to_csv('/content/drive/My Drive/clean/merge/draft1.csv', index=False)  # Update the file name accordingly

# Print completion message
print("Finished processing.")


Processed Aswath Damodaran
Skipping row 1 as URL is empty or 'not found'.
Processed Prof. (Dr) A.V Arun Kumar
Skipping row 3 as URL is empty or 'not found'.
Skipping row 4 as URL is empty or 'not found'.
Skipping row 5 as URL is empty or 'not found'.
Skipping row 6 as URL is empty or 'not found'.
Skipping row 7 as URL is empty or 'not found'.
Processed Prof Dr A M Jose ASE AUH
Skipping row 9 as URL is empty or 'not found'.
Processed A. Narayanamoorthy
Skipping row 11 as URL is empty or 'not found'.
Processed A. Pramanick
Skipping row 13 as URL is empty or 'not found'.
Skipping row 14 as URL is empty or 'not found'.
Skipping row 15 as URL is empty or 'not found'.
Skipping row 16 as URL is empty or 'not found'.
Skipping row 17 as URL is empty or 'not found'.
Skipping row 18 as URL is empty or 'not found'.
Processed ACHAL GAUR
Processed ADITI CHAUBAL
Processed AISWARYA THOMAS
Processed AJIT KUMAR
Processed AKHIL ILANGO
Processed AMARESH SAMANTARAYA
Skipping row 25 as URL is empty or 'not 

### Visualize a little bit of the collected data

In [ ]:
# Recheck on the total number of rows or subjects
total_rows = df.shape[0]
print(f'Total number of rows for this batch: {total_rows}')

Total number of rows for this batch: 2636


Check if the all the columns are listed

In [ ]:
print(df.columns.tolist())


['Name', 'Institution', 'scholar_profile_url', 'Personal website', 'LinkedIn profile', 'University profile link', 'Research Gate', 'UniversityType', 'revised_gender', 'college_names', 'archive_link', 'college_url', 'State', 'Specialisation', 'AreaofInterest', 'UGDegree', 'UGDepartment', 'UGInstitute', 'PGDegree', 'PGDepartment', 'PGInstitute', 'MPhil', 'MPhilDepartment', 'MPhilInstitute', 'Doctoral', 'DoctoralDepartment', 'DoctoralInstitute', 'FirstJobPosition', 'FirstJobInstitution', 'FirstJobDuration', 'SecondJobPosition', 'SecondJobInstitution', 'SecondJobDuration', 'ThirdJobPosition', 'ThirdJobInstitution', 'ThirdJobDuration', 'FourthJobPosition', 'FourthJobInstitution', 'FourthJobDuration', 'FifthJobPosition', 'FifthJobInstitution', 'FifthJobDuration', 'SixthJobPosition', 'SixthJobInstitution', 'SixthJobDuration', 'SeventhJobPosition', 'SeventhJobInstitution', 'SeventhJobDuration', 'EightJobPosition', 'EightJobInstitution', 'EightJobDuration', 'NinthJobPosition', 'NinthJobInstitut

Check on the columns of details of Google scholar profiles

In [ ]:
# Specify the columns you want to view
columns_to_view = ['scholar_profile_url','names_scholar', 'college_names', 'position', 'field', 'total_citation', 'h-index', 'i10-index', 'profile_since', 'co-authors', 'citations_5y', 'h-index_5y', 'i10-index_5y']

# Display the first 50 rows of the specified columns
df[columns_to_view].head(50)

,scholar_profile_url,names_scholar,college_names,position,field,total_citation,h-index,i10-index,profile_since,co-authors,citations_5y,h-index_5y,i10-index_5y
0,https://scholar.google.com/citations?user=NF8i...,Anush Sankaran,Pondicherry University,Microsoft Security Research,"AI for Security, Deep Learning, Democratizatio...",1332,21,27,Since 2019,"Mayank VatsaProfessor, IIT JodhpurVerified ema...",1064,20,26
1,https://scholar.google.com/citations?user=3JHH...,Dr. Aas Mohammad,Jamia Millia Islamia,Professor in Mechanical Engineering,Mechanism and Machine Design,873,15,20,Since 2019,"Dr. Pallav Gupta (AMIE, LMMRSI, LMT...Dept. of...",613,12,12
2,https://scholar.google.com/citations?user=nwtU...,Aasheerwad Dwivedi,Delhi University,Shri Ram College of Commerec and Jawaharlal Ne...,economics,25,4,0,Since 2019,not found,23,4,0
3,https://scholar.google.com/citations?user=TZDO...,Abhiman Das,IIM-Ahmedabad,Professor of Economics,Economics,3836,26,40,Since 2019,not found,1431,17,27
4,https://scholar.google.com/citations?user=OcWI...,Abhinaba Lahiri,IIT Jodhpur,Assistant professor,"social choice theory, mechanism design, game t...",37,3,2,Sinds 2019,"Medeauteurs, Anup PramanikShiv Nadar Universit...",35,3,2
5,https://scholar.google.com/citations?user=sMZQ...,Abhinav Anand,IIM-Bangalore,"Postdoctoral researcher, Duke University","Quantum Computation, Quantum Information, Erro...",1923,8,8,Siden 2019,"MedforfattereSe alle, Alan Aspuru-GuzikProfess...",1914,8,8
6,https://scholar.google.com/citations?user=BImk...,Abhishek Samantray,IIT Roorkee,Indian Institute of Technology Roorkee,"applied data science, social networks, big dat...",83,3,1,Since 2019,Paolo PinDipartimento di Economia Politica e S...,82,3,1
7,https://scholar.google.com/citations?user=6KSg...,Abhradeep Maiti,IIM-Kashipur,Indian Institute of Management Kashipur,Applied Microeconomics,38,4,1,Since 2019,"Anthon EffProfessor of Economics, Middle Tenne...",29,3,1
8,https://scholar.google.com/citations?user=FMBD...,Aditya Shrinivas,IIM-Bangalore,"Assistant Professor, IIM Bangalore","Development Economics, Food Security, Risk Sha...",854,4,3,Since 2019,"Rachel Bezner KerrProfessor, Department of Glo...",563,4,3
9,https://scholar.google.com/citations?user=T-jw...,Aditya Vikram Hardikar,Indian Institute of Technology Kanpur,"PhD Candidate, New York University",Soft Condensed Matter,not found,not found,not found,not found,not found,not found,not found,not found
